# Zero-Shot Learning untuk Sentiment Analysis

**Deskripsi**: Tutorial ini mendemonstrasikan *zero-shot classification* — teknik mengklasifikasikan sentimen teks **tanpa pelatihan (training)** sama sekali. Model yang digunakan adalah `facebook/bart-large-mnli` yang telah dilatih untuk tugas *natural language inference* (NLI) dan dapat langsung digunakan untuk klasifikasi tanpa data training.

**Tujuan Pembelajaran**:
- Memahami konsep *zero-shot learning* dan bagaimana ia berbeda dari fine-tuning
- Mampu menggunakan pipeline zero-shot classification dari Hugging Face
- Mampu mengevaluasi hasil zero-shot pada dataset sentimen Bahasa Indonesia
- Mampu membandingkan performa zero-shot vs fine-tuning
- Memahami kapan harus menggunakan zero-shot vs fine-tuning

---## 1. Instalasi Dependensi

In [ ]:
!pip install transformers datasets scikit-learn seaborn matplotlib pandas tqdm torch

---## 2. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

import torch
from transformers import pipeline
from datasets import load_dataset

import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---## 3. Konsep Zero-Shot Learning

**Zero-shot classification** bekerja melalui pendekatan *Natural Language Inference (NLI)*:

1. Kita memiliki sebuah teks dan sebuah *hypothesis template* (mis: "This text expresses [NEGATIVE] sentiment")
2. Model NLI memeriksa apakah teks **entails** (mendukung) atau **contradicts** (bertentangan dengan) hypothesis tersebut
3. Label dengan skor entailment tertinggi menjadi prediksi

**Keuntungan**:
- Tidak perlu data training
- Bisa langsung digunakan untuk label kustom apapun
- Cocok untuk eksplorasi cepat

**Kekurangan**:
- Performa umumnya lebih rendah dari fine-tuning
- Lebih lambat karena harus mengevaluasi setiap pasangan teks-label
- Sensitif terhadap pemilihan kata label dan hypothesis template

---## 4. Load Dataset

In [ ]:
# Load dataset sentimen Bahasa Indonesia
dataset = load_dataset("sepidmnorozy/Indonesian_sentiment")

print(f"Dataset splits: {dataset.keys()}")
print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print(f"Test size: {len(dataset['test'])}")

In [ ]:
label_map = {0: "NEGATIVE", 1: "POSITIVE"}

print("=== Contoh Data ===")
for i in range(5):
    sample = dataset['test'][i]
    print(f"[{i}] Label: {label_map[sample['label']]}")
    print(f"    Teks: {sample['text'][:120]}...")
    print()

---## 5. Load Zero-Shot Pipeline

In [ ]:
# Load zero-shot classification pipeline
# Model NLI multilingual: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
# Model alternatif: facebook/bart-large-mnli (hanya English tapi lebih ringan)

MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device=device
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

---## 6. Uji Coba Zero-Shot

In [ ]:
# Candidate labels dalam Bahasa Inggris (model ini berbasis English NLI)
candidate_labels = ["positive", "negative"]
hypothesis_template = "This text expresses {} sentiment."

# Uji dengan beberapa contoh
test_texts = [
    "makanan di restoran ini sangat enak dan pelayanannya ramah",
    "produknya jelek banget, tidak sesuai dengan deskripsi, mengecewakan",
    "barangnya biasa aja sih, standar",
    "recommended banget! puas dengan pelayanannya",
    "pelayanan sangat lambat dan makanannya tidak enak"
]

print("=== Zero-Shot Predictions ===\n")
for text in test_texts:
    result = classifier(text, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False)
    print(f"Teks: {text}")
    print(f"Label: {result['labels'][0]} (skor: {result['scores'][0]:.3f})")
    print(f"Skor detail: neg={result['scores'][1]:.3f}, pos={result['scores'][0]:.3f}")
    print()

---## 7. Eksperimen dengan Berbagai Hypothesis Template

Salah satu keunikan zero-shot adalah kita bisa bermain dengan *hypothesis template* untuk melihat pengaruhnya terhadap hasil.

In [ ]:
# Bandingkan beberapa hypothesis template
templates = [
    "This text expresses {} sentiment.",
    "The sentiment of this text is {}.",
    "This is a {} review.",
    "The writer feels {} about this.",
    "{}"
]

sample_text = "makanan di restoran ini sangat enak dan pelayanannya ramah"

print(f"Teks: {sample_text}\n")
print(f"{'Template':<65} {'Prediksi':<12} {'Skor':<8}")
print("=" * 85)

for template in templates:
    result = classifier(sample_text, candidate_labels, hypothesis_template=template, multi_label=False)
    print(f"{template:<65} {result['labels'][0]:<12} {result['scores'][0]:.3f}")

---## 8. Evaluasi pada Seluruh Test Set

Kita akan mengevaluasi performa zero-shot pada subset test (100 sampel) karena proses zero-shot cukup lambat.

In [ ]:
# Ambil subset test untuk evaluasi (100 sampel karena zero-shot lambat)
test_subset = dataset['test'].select(range(100))
true_labels = [sample['label'] for sample in test_subset]
texts = [sample['text'] for sample in test_subset]

print(f"Evaluating {len(texts)} samples with zero-shot...")
print("This may take a while...\n")

In [ ]:
# Jalankan prediksi zero-shot
start_time = time.time()
pred_labels = []

for i, text in enumerate(texts):
    result = classifier(text, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False)
    pred = 0 if result['labels'][0] == 'negative' else 1
    pred_labels.append(pred)
    
    if (i + 1) % 20 == 0:
        print(f"  Processed {i+1}/{len(texts)} samples...")

elapsed = time.time() - start_time
print(f"\nDone! Elapsed: {elapsed:.2f}s ({elapsed/len(texts):.2f}s per sample)")

---## 9. Metrik Evaluasi

In [ ]:
# Hitung metrik
accuracy = accuracy_score(true_labels, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='macro')

print("=== Zero-Shot Classification Results (100 samples) ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f} (macro)")
print(f"Recall:    {recall:.4f} (macro)")
print(f"F1-Score:  {f1:.4f} (macro)")
print()
print(classification_report(true_labels, pred_labels, target_names=["NEGATIVE", "POSITIVE"]))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(true_labels, pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['NEGATIVE', 'POSITIVE'],
            yticklabels=['NEGATIVE', 'POSITIVE'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Zero-Shot Sentiment Classification')
plt.show()

---## 10. Perbandingan: Zero-Shot vs Model yang Belum Dilatih

In [ ]:
# Coba tanpa fine-tuning: gunakan model BERT mentah + linear layer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

raw_model_name = "bert-base-uncased"
raw_model = AutoModelForSequenceClassification.from_pretrained(raw_model_name, num_labels=2)
raw_tokenizer = AutoTokenizer.from_pretrained(raw_model_name)

print("=== Perbandingan: BERT Mentah (random weights) vs Zero-Shot ===")
print(f"Nama model zero-shot: {MODEL_NAME}")
print(f"Nama model raw: {raw_model_name}")
print("\nZero-shot menggunakan model NLI yang sudah dilatih untuk memahami hubungan antar teks.")
print("Model BERT mentah dengan random head tidak bisa melakukan klasifikasi tanpa training.")
print("\nInilah KEUNGGULAN zero-shot: langsung bisa digunakan tanpa data training!")

---## 11. Zero-Shot dengan Label Berbahasa Indonesia

Mari kita coba apakah model bisa memahami label dalam Bahasa Indonesia.

In [ ]:
# Coba dengan label Bahasa Indonesia
indonesian_labels = ["positif", "negatif"]

sample_text = "makanan di restoran ini sangat enak dan pelayanannya ramah"

print("=== Label Bahasa Inggris vs Bahasa Indonesia ===")

for lang, labels in [("English", candidate_labels), ("Indonesia", indonesian_labels)]:
    result = classifier(sample_text, labels, multi_label=False)
    print(f"\n{lang}: {labels}")
    print(f"  Prediksi: {result['labels'][0]} (skor: {result['scores'][0]:.3f})")
    print(f"  Semua skor:")
    for label, score in zip(result['labels'], result['scores']):
        print(f"    {label}: {score:.3f}")

---## 12. Analisis Error

Mari kita lihat contoh-contoh di mana zero-shot gagal untuk memahami pola kesalahannya.

In [ ]:
# Identifikasi misklasifikasi
misclassified = []
for i, (true, pred) in enumerate(zip(true_labels, pred_labels)):
    if true != pred:
        misclassified.append((texts[i], true, pred))

print(f"Jumlah misklasifikasi: {len(misclassified)} dari {len(texts)} sampel\n")

print("=== Contoh Misklasifikasi Zero-Shot ===")
for i, (text, true, pred) in enumerate(misclassified[:10]):
    print(f"[{i+1}] Teks: {text[:100]}...")
    print(f"    True: {label_map[true]} | Pred: {label_map[pred]}")
    print()

In [ ]:
# Kategorisasi error
print("=== Pola Umum Kesalahan Zero-Shot ===")
print("\n1. Teks ambigu (netral diprediksi sebagai salah satu sentimen)")
print("2. Sarkasme (teks positif tapi sebenarnya negatif / sebaliknya)")
print("3. Teks pendek tanpa cukup konteks")
print("4. Campuran sentimen dalam satu teks")
print("\nBandingkan dengan fine-tuning yang bisa belajar pola spesifik dataset!")

---## 13. Kesimpulan

Dalam tutorial zero-shot learning ini, kita telah mempelajari:

1. ✅ **Konsep zero-shot learning** — klasifikasi tanpa data training
2. ✅ **Pipeline Hugging Face** — menggunakan model NLI untuk zero-shot
3. ✅ **Hypothesis template** — pengaruh template terhadap hasil prediksi
4. ✅ **Evaluasi** — metrik dan confusion matrix
5. ✅ **Analisis error** — memahami kelemahan zero-shot

**Kapan menggunakan Zero-Shot?**
- Eksplorasi cepat label baru
- Tidak ada data training
- Label berubah-ubah / dinamis
- Prototyping sebelum investasi fine-tuning

**Kapan menggunakan Fine-Tuning?**
- Performa maksimal dibutuhkan
- Data training tersedia (1000+ sampel)
- Domain spesifik dengan vocabulary unik
- Produksi / deployment